# FIREGUARD ID v2 — Google Colab Quickstart & Reproducibility Entry Point

**Pilot Area:** Riau Province, Indonesia | **SDG 13 Target 13.1**  
**Methodology:** Unsupervised Spatial Clustering (K-Means, HDBSCAN, DEC) + MILP Resource Optimization

This notebook detects whether it is running in Google Colab or locally, installs Colab-compatible dependencies without breaking pre-installed CUDA PyTorch, verifies hardware (GPU/CPU fallback), configures optional Google Drive persistence, runs diagnostics and pipeline stages, and documents secure Streamlit preview access.

In [ ]:
import os, sys, subprocess
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')
print('Python:', sys.version)
print('Running in Google Colab:', IN_COLAB)

if IN_COLAB and not Path('/content/FIREGUARD-ID').exists():
    subprocess.run(['git', 'clone', '-b', 'prototype-v2', 'https://github.com/Ghof-AM/FIREGUARD-ID.git', '/content/FIREGUARD-ID'], check=True)
if IN_COLAB:
    os.chdir('/content/FIREGUARD-ID')
print('Working Directory:', os.getcwd())

In [ ]:
# Install Colab-safe dependencies (preserves Colab pre-installed CUDA torch)
if IN_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-colab.txt'], check=True)
print('Dependency check complete.')

In [ ]:
# Optional Google Drive persistence (set USE_DRIVE = True in Colab to persist outputs across sessions)
USE_DRIVE = False
if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    drive_out = Path('/content/drive/MyDrive/FIREGUARD-ID/outputs')
    drive_out.mkdir(parents=True, exist_ok=True)
    os.environ['FIREGUARD_OUTPUT_DIR'] = str(drive_out)
    print('Persisting outputs to:', os.environ['FIREGUARD_OUTPUT_DIR'])
else:
    print('Using repository-local data/ and outputs/ directories.')

In [ ]:
# Hardware, CUDA/CPU fallback & Package Verification
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
from src.utils import get_hardware_diagnostics
import json
diag = get_hardware_diagnostics()
print(json.dumps(diag, indent=2))

In [ ]:
# Verify dataset cache & run end-to-end pipeline (or selected stages)
# Note: DEC runs on CUDA GPU if available (~20-40s on T4) or automatically falls back to CPU (~2-4m).
from src.application.pipeline import run_pipeline
manifest = run_pipeline(stages=['data', 'features', 'eda', 'vulnerability', 'prioritization', 'visualization'])
print('Pipeline stages executed:', manifest['stages_executed'])
print('Timings (s):', manifest['timings_seconds'])

## Optional: Full Clustering + Optimization Re-training & Secure Streamlit Launch in Colab
To re-run all three models (`K-Means`, `HDBSCAN`, `DEC`) and all 6 MILP optimization modules from scratch:
```python
run_pipeline(stages=['all'])  # or python -m src.application.pipeline --stages all
```
To preview the unified 11-page Streamlit app from Google Colab without exposing credentials:
```bash
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared
!nohup streamlit run src/application/app.py --server.port 8501 --server.headless true > streamlit.log 2>&1 &
!./cloudflared tunnel --url http://localhost:8501
```